# DeepBaseEditor efficiency: 100 trials from saved TXT splits

Run the split notebook first and set the same `EDITOR_MODE` here.

In [ ]:
from pathlib import Path
import hashlib, json, random, warnings
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error
from torch.utils.data import Dataset, DataLoader

EDITOR_MODE = "ABE_Efficiency"
# EDITOR_MODE = "CBE_Efficiency"
# EDITOR_MODE = "CBE_Efficiency_CA"

OPTUNA_SEED = 42
MODEL_SEED = 42
N_TRIALS = 100
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

BASE_OUTPUT_DIR = Path(f"results/deepbaseeditor_{EDITOR_MODE.lower()}_fixed_split")
SPLIT_DIR = BASE_OUTPUT_DIR / "saved_splits"
RESULTS_DIR = BASE_OUTPUT_DIR / "trial_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = SPLIT_DIR / "train_split.csv"
VALIDATION_FILE = SPLIT_DIR / "validation_split.csv"
UNSEEN_FILE = SPLIT_DIR / "unseen_split.csv"
MANIFEST_FILE = BASE_OUTPUT_DIR / "split_manifest.json"

for p in [TRAIN_FILE, VALIDATION_FILE, UNSEEN_FILE, MANIFEST_FILE]:
    assert p.exists(), f"Missing {p.resolve()}"

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("Device:", DEVICE)


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

with open(MANIFEST_FILE) as f:
    manifest = json.load(f)

if manifest["editor_mode"] != EDITOR_MODE:
    raise ValueError("EDITOR_MODE does not match the saved manifest.")

assert sha256(TRAIN_FILE) == manifest["train_sha256"]
assert sha256(VALIDATION_FILE) == manifest["validation_sha256"]
assert sha256(UNSEEN_FILE) == manifest["unseen_sha256"]

train_data = pd.read_csv(TRAIN_FILE)
validation_data = pd.read_csv(VALIDATION_FILE)
unseen_data = pd.read_csv(UNSEEN_FILE)

USE_CHROMATIN = EDITOR_MODE == "CBE_Efficiency_CA"
print(len(train_data), len(validation_data), len(unseen_data))


In [ ]:
BASE_TO_INDEX = {"A":0, "C":1, "G":2, "T":3}
MODEL_INPUT_LENGTH = 24

def encode(series):
    X = np.zeros((len(series), 4, MODEL_INPUT_LENGTH), dtype=np.float32)
    for i, context in enumerate(series.astype(str)):
        context = context.strip().upper()
        if len(context) != 30 or not set(context).issubset(BASE_TO_INDEX):
            raise ValueError(f"Invalid context at row {i}: {context}")
        for j, base in enumerate(context[:MODEL_INPUT_LENGTH]):
            X[i, BASE_TO_INDEX[base], j] = 1.0
    return X

X_train = encode(train_data["target_context_30nt"])
X_validation = encode(validation_data["target_context_30nt"])
X_unseen = encode(unseen_data["target_context_30nt"])

y_train = train_data["activity"].to_numpy(np.float32)
y_validation = validation_data["activity"].to_numpy(np.float32)
y_unseen = unseen_data["activity"].to_numpy(np.float32)

if USE_CHROMATIN:
    ca_train = train_data["chromatin_accessibility"].to_numpy(np.float32)
    ca_validation = validation_data["chromatin_accessibility"].to_numpy(np.float32)
    ca_unseen = unseen_data["chromatin_accessibility"].to_numpy(np.float32)
else:
    ca_train = np.zeros(len(train_data), dtype=np.float32)
    ca_validation = np.zeros(len(validation_data), dtype=np.float32)
    ca_unseen = np.zeros(len(unseen_data), dtype=np.float32)

print(X_train.shape, X_validation.shape, X_unseen.shape)


In [ ]:
class BEDataset(Dataset):
    def __init__(self, X, ca, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.ca = torch.tensor(ca, dtype=torch.float32).view(-1, 1)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.ca[i], self.y[i]

class DeepBaseEditorEfficiency(nn.Module):
    def __init__(
        self,
        conv_filters=96,
        kernel_size=5,
        dense_units=80,
        dropout=0.3,
        use_chromatin=False,
        chromatin_hidden=32,
    ):
        super().__init__()
        self.use_chromatin = use_chromatin
        self.conv = nn.Conv1d(4, conv_filters, kernel_size)

        with torch.no_grad():
            flat = self.conv(torch.zeros(1, 4, MODEL_INPUT_LENGTH)).flatten(1).shape[1]

        self.sequence_fc = nn.Linear(flat, dense_units)

        if use_chromatin:
            self.chromatin_fc = nn.Linear(1, chromatin_hidden)
            self.merge_fc = nn.Linear(dense_units + chromatin_hidden, dense_units)

        self.out = nn.Linear(dense_units, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, ca):
        x = self.dropout(F.relu(self.conv(x)).flatten(1))
        x = self.dropout(F.relu(self.sequence_fc(x)))

        if self.use_chromatin:
            ca = F.relu(self.chromatin_fc(ca * 100.0))
            x = self.dropout(F.relu(self.merge_fc(torch.cat([x, ca], dim=1))))

        return self.out(x).view(-1)


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def safe_pearson(y, p):
    if len(y) < 2 or np.std(y) == 0 or np.std(p) == 0:
        return np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(pearsonr(y, p)[0])

def safe_spearman(y, p):
    if len(y) < 2 or np.std(y) == 0 or np.std(p) == 0:
        return np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(spearmanr(y, p)[0])

def predict(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for X, ca, y in loader:
            p = model(X.to(DEVICE), ca.to(DEVICE))
            ys.extend(y.numpy())
            ps.extend(p.cpu().numpy())
    return np.asarray(ys), np.asarray(ps)

train_dataset = BEDataset(X_train, ca_train, y_train)
validation_dataset = BEDataset(X_validation, ca_validation, y_validation)
unseen_dataset = BEDataset(X_unseen, ca_unseen, y_unseen)


In [ ]:
trial_records = []
trial_states = {}

def objective(trial):
    set_seed(MODEL_SEED)

    arch = {
        "conv_filters": trial.suggest_int("conv_filters", 48, 192, step=16),
        "kernel_size": trial.suggest_categorical("kernel_size", [3,5,7]),
        "dense_units": trial.suggest_int("dense_units", 48, 192, step=16),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "chromatin_hidden": trial.suggest_int("chromatin_hidden", 16, 64, step=16),
    }
    lr = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16,32,64])
    weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True)

    g = torch.Generator().manual_seed(MODEL_SEED)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, generator=g
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=batch_size, shuffle=False
    )
    unseen_loader = DataLoader(
        unseen_dataset, batch_size=batch_size, shuffle=False
    )

    model = DeepBaseEditorEfficiency(
        **arch, use_chromatin=USE_CHROMATIN
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    criterion = nn.MSELoss()

    best_mse = np.inf
    best_state = None
    best_epoch = 0
    counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for X, ca, y in train_loader:
            X, ca, y = X.to(DEVICE), ca.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X, ca), y)
            loss.backward()
            optimizer.step()

        val_y, val_p = predict(model, validation_loader)
        val_mse = float(mean_squared_error(val_y, val_p))
        trial.report(val_mse, epoch)

        if val_mse < best_mse:
            best_mse = val_mse
            best_epoch = epoch
            counter = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            counter += 1

        if counter >= EARLY_STOPPING_PATIENCE:
            break

    model.load_state_dict(best_state)
    model.to(DEVICE)

    unseen_y, unseen_p = predict(model, unseen_loader)
    up = safe_pearson(unseen_y, unseen_p)
    us = safe_spearman(unseen_y, unseen_p)

    trial.set_user_attr("best_epoch", int(best_epoch))
    trial_records.append({
        "trial": trial.number,
        "validation_mse": best_mse,
        "unseen_pearson": up,
        "unseen_spearman": us,
    })
    trial_states[trial.number] = best_state

    print(
        f"Trial {trial.number:3d} | Validation MSE: {best_mse:.6f} | "
        f"Unseen Pearson: {up:.4f} | Unseen Spearman: {us:.4f} | "
        f"Epoch: {best_epoch}"
    )
    return best_mse

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
)
study.optimize(objective, n_trials=N_TRIALS)


In [ ]:
results_df = pd.DataFrame(trial_records).sort_values("trial").reset_index(drop=True)

results_df.to_csv(RESULTS_DIR / "all_100_trial_metrics.csv", index=False)
results_df["validation_mse"].to_csv(
    RESULTS_DIR / "Validation_loss.txt", index=False, header=False
)
results_df["unseen_pearson"].to_csv(
    RESULTS_DIR / "Unseen_Pearson.txt", index=False, header=False
)
results_df["unseen_spearman"].to_csv(
    RESULTS_DIR / "Unseen_Spearman.txt", index=False, header=False
)

best_trial = study.best_trial.number
best_params = study.best_trial.params
arch = {k: best_params[k] for k in [
    "conv_filters", "kernel_size", "dense_units", "dropout", "chromatin_hidden"
]}

best_model = DeepBaseEditorEfficiency(
    **arch, use_chromatin=USE_CHROMATIN
)
best_model.load_state_dict(trial_states[best_trial])

best_row = results_df.loc[results_df["trial"] == best_trial].iloc[0]

torch.save({
    "editor_mode": EDITOR_MODE,
    "use_chromatin": USE_CHROMATIN,
    "model_state_dict": best_model.state_dict(),
    "best_trial": int(best_trial),
    "best_params": best_params,
    "best_epoch": int(study.best_trial.user_attrs["best_epoch"]),
    "validation_mse": float(best_row["validation_mse"]),
    "unseen_pearson": float(best_row["unseen_pearson"]),
    "unseen_spearman": float(best_row["unseen_spearman"]),
}, RESULTS_DIR / "best_DeepBaseEditor_model.pt")

with open(RESULTS_DIR / "best_trial_summary.json", "w") as f:
    json.dump({
        "editor_mode": EDITOR_MODE,
        "best_trial": int(best_trial),
        "best_params": best_params,
        "best_epoch": int(study.best_trial.user_attrs["best_epoch"]),
        "validation_mse": float(best_row["validation_mse"]),
        "unseen_pearson": float(best_row["unseen_pearson"]),
        "unseen_spearman": float(best_row["unseen_spearman"]),
    }, f, indent=2)

display(results_df.head())
print("Best trial:", best_trial)
print(best_row)
print("Saved to:", RESULTS_DIR.resolve())
